# 04 — Research Conclusions

**Summary of all findings from the OFI research pipeline.**

1. OFI signal summary across all variants
2. Statistical significance of price impact
3. Which LOB levels matter most?
4. Integrated vs best-level OFI comparison
5. Intraday OFI patterns
6. Key findings and implications

In [ ]:
import importlib.util, os, pickle
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, os.path.abspath(path))
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

ofi_mod = load_module('ofi',          '../src/ofi.py')
pi_mod  = load_module('price_impact', '../src/price_impact.py')
viz_mod = load_module('viz',          '../src/visualization.py')

with open('../data/ofi_data.pkl', 'rb') as f:
    data = pickle.load(f)

df         = data['df']
level_cols = data['level_cols']
weights    = data['weights']
evr        = data['evr']
agg_1s     = data['agg_results']['1s']
agg_5s     = data['agg_results']['5s']
agg_30s    = data['agg_results']['30s']

print(f'Data loaded. {len(df)} tick events, {len(agg_5s)} 5s intervals.')

## 1. OFI Signal Summary

In [ ]:
print('=== OFI SIGNAL SUMMARY ===')
print(f'Symbol:           AAPL')
print(f'Date:             {df.ts_event.iloc[0].date()}')
print(f'Time window:      {df.ts_event.iloc[0].strftime("%H:%M:%S")} - {df.ts_event.iloc[-1].strftime("%H:%M:%S")}')
print(f'Total events:     {len(df):,}')
print(f'LOB levels used:  {len(level_cols)}')
print(f'PCA variance:     {evr*100:.1f}% explained by first PC')
print()

for col in ['ofi_best', 'ofi_integrated']:
    if col not in agg_5s.columns:
        continue
    s = agg_5s[col].dropna()
    print(f'{col}:')
    print(f'  Mean:     {s.mean():.4f}')
    print(f'  Std:      {s.std():.4f}')
    print(f'  Skew:     {s.skew():.4f}')
    print(f'  Kurtosis: {s.kurtosis():.4f}')
    print()

## 2. Price Impact Summary — All Frequencies

In [ ]:
print('=== PRICE IMPACT ACROSS AGGREGATION FREQUENCIES ===')
print('Model: Δp_{t+1} = α + β×OFI_t  (Newey-West standard errors)')
print()

results_summary = []
for freq, agg_df in [('1s', agg_1s), ('5s', agg_5s), ('30s', agg_30s)]:
    for ofi_col in ['ofi_best', 'ofi_integrated']:
        if ofi_col not in agg_df.columns or 'fwd_return' not in agg_df.columns:
            continue
        res = pi_mod.ofi_price_impact(agg_df, ofi_col=ofi_col, return_col='fwd_return')
        if 'error' not in res:
            results_summary.append({
                'Frequency': freq,
                'OFI Type':  ofi_col,
                'β':         res['beta'],
                't-stat':    res['t_stat_nw'],
                'p-value':   res['p_value_nw'],
                'R²':        res['r_squared'],
                'Sig.':      '✓' if res['significant'] else '✗',
                'N':         res['n_obs'],
            })

summary_df = pd.DataFrame(results_summary)
if not summary_df.empty:
    display(summary_df.style.background_gradient(subset=['R²'], cmap='YlGn'))

    # Bar chart comparison
    fig = go.Figure()
    for ofi_type in summary_df['OFI Type'].unique():
        sub = summary_df[summary_df['OFI Type'] == ofi_type]
        fig.add_trace(go.Bar(
            x=sub['Frequency'], y=sub['R²'],
            name=ofi_type, opacity=0.85
        ))
    fig.update_layout(
        title='OFI Price Impact R² by Frequency',
        xaxis_title='Aggregation Frequency',
        yaxis_title='R² (price impact)',
        barmode='group', height=400, template='plotly_white'
    )
    fig.show()

## 3. LOB Level Contribution

In [ ]:
level_cols_in_agg = [c for c in level_cols if c in agg_5s.columns]

if level_cols_in_agg:
    level_r2 = pi_mod.level_r2_decomposition(agg_5s, level_cols_in_agg, 'fwd_return')
    print('R² contribution by LOB level:')
    display(level_r2)
    fig2 = viz_mod.level_r2_chart(level_r2)
    fig2.show()

# PCA weights
fig3 = viz_mod.pca_weights_chart(weights, level_cols, evr)
fig3.show()
print('\nPCA weights show which levels dominate the integrated signal.')
print('Positive weight = that level moves in the same direction as the dominant flow.')

## 4. Intraday OFI Patterns

In [ ]:
# How does OFI signal strength vary through the trading day?
if 'ofi_best' in agg_5s.columns and 'fwd_return' in agg_5s.columns:
    agg_5s['hour'] = pd.to_datetime(agg_5s['ts_event']).dt.hour
    agg_5s['minute'] = pd.to_datetime(agg_5s['ts_event']).dt.minute

    # Rolling OFI impact
    rolling_df = pi_mod.rolling_price_impact(
        agg_5s, 'ofi_best', 'fwd_return', window=30
    )
    fig4 = viz_mod.rolling_impact_chart(rolling_df)
    fig4.show()

    # OFI magnitude over time
    ofi_abs = agg_5s['ofi_best'].abs().rolling(20).mean()
    fig5 = go.Figure()
    fig5.add_trace(go.Scatter(
        x=agg_5s['ts_event'], y=ofi_abs,
        line=dict(color='#7950F2', width=2),
        fill='tozeroy', fillcolor='rgba(121,80,242,0.1)',
        name='|OFI| (20-period MA)'
    ))
    fig5.update_layout(
        title='OFI Magnitude Over Trading Day (Rolling 20-period MA)',
        xaxis_title='Time', yaxis_title='|OFI|',
        height=350, template='plotly_white'
    )
    fig5.show()
    print('OFI activity typically spikes around news, earnings, or large institutional orders.')

---
## 5. Research Findings

### What this project demonstrates

This OFI research pipeline implements the full framework from **Cont, Kukanov & Stoikov (2014)** — the seminal paper on order flow imbalance that is cited in most quantitative trading research on microstructure.

### Key findings

**1. OFI has statistically significant price impact.** The regression Δp_{t+1} = α + β×OFI_t with Newey-West standard errors shows that OFI predicts short-term mid-price changes. The use of HAC standard errors (not naive OLS) is critical — raw tick data has significant autocorrelation that inflates t-statistics.

**2. Integrated OFI (PCA) outperforms best-level OFI.** The first principal component of multi-level OFI explains a large fraction of the variance and has higher R² in price impact regressions. This confirms Cont et al.'s finding that deeper book levels add incremental information.

**3. Impact decays but has a permanent component.** The multi-horizon analysis reveals the trade-off between temporary (noise) and permanent (information) impact. Aggressive orders that move the price permanently carry more information than orders that revert.

**4. Price impact is time-varying.** The rolling regression shows that the β coefficient changes through the day, consistent with the literature showing that market microstructure is non-stationary.

### Methodology improvements vs original notebook

| Original | This version |
|---|---|
| Python loop over rows (slow) | Vectorized NumPy (fast) |
| No price impact regression | Full OLS + Newey-West |
| No statistical tests | t-stats, p-values, R² |
| No temporal aggregation | 1s / 5s / 30s intervals |
| No visualization | LOB heatmap, decay charts, rolling β |
| No permanent/temporary decomposition | Full decomposition |

### What I'd do next with more data
- **Cross-asset OFI**: With LOB data for MSFT, NVDA, SPY alongside AAPL, test whether AAPL OFI predicts MSFT price changes (cross-impact)
- **Intraday seasonality**: With multiple days of data, test whether OFI predictability is higher near open/close
- **Machine learning**: Train a model that uses the full OFI vector at multiple lags to predict direction — this is the basis of HFT alpha models
- **Adverse selection**: Measure whether trades that follow high-OFI signals move price against market makers more than random trades